# Project 3 — Smart Waste Sorting

This starter uses RealWaste images to classify the dominant visible waste into nine categories. It predicts a **visual waste category, not the correct disposal bin**. Packaging status, contamination, composition, separability, and local regulations may all affect disposal decisions and may not be visible in an image. Refer to the project brief for the complete objectives.

RealWaste is provided by Single, Iranmanesh, and Raad through the UCI Machine Learning Repository under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).

### Requirements
```bash
python -m pip install --upgrade \
    "torch==2.9.1" \
    "torchvision==0.24.1" \
    --index-url https://download.pytorch.org/whl/cu128
```

In [ ]:
import sys
import torch
from importlib.metadata import version

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("Torchvision:", version("torchvision"))
print("PyTorch CUDA:", torch.version.cuda)

## 1. Your Tasks

The provided notebook establishes a simple baseline for detecting potentially toxic comments. Your task is to **evaluate this baseline, identify its limitations, and investigate possible improvements**.

### 1.1 Explore the Dataset

Get familiar with the data and the classification task.

- Examine the dataset characteristics, class distribution, and representative examples.
- Consider how the target labels are defined and whether the data is suitable for the intended application.
- Identify potential issues related to data quality, coverage, or representativeness.

### 1.2 Evaluate the Baseline

Determine how well the model performs and which evaluation criteria are appropriate.

- Select and justify metrics relevant to the content moderation task.
- Evaluate overall and class-wise performance.
- Consider the consequences of different types of errors, particularly false positives and false negatives.

### 1.3 Analyze Errors and Diagnose Limitations

Investigate where and why the classifier makes mistakes.

- Examine misclassified examples and identify recurring patterns.
- Formulate possible explanations for the observed errors.
- Distinguish observations supported by the data from hypotheses that require further investigation.

### 1.4 Investigate Possible Improvements

Based on your observations and relevant literature, identify promising directions for improving the baseline.

- Research alternative approaches, representations, or model configurations.
- Explain which limitations these approaches might address.
- Where feasible, implement and evaluate an improvement against the original baseline using a consistent evaluation setup.

### 1.5 Document Your Findings

Summarize your experiments and the reasoning behind your decisions.

- What did you investigate, and why?
- What evidence supports your observations and conclusions?
- Which improvements appear promising, and what remains unresolved?

**Important:** Your decisions should be grounded in **empirical evidence or relevant research**, rather than assumptions. When proposing changes, explain the motivation and evaluate their effects whenever possible.

Use the validation set for model development and comparisons. Keep the test set reserved for final evaluation.



## 2. Starter Code


### 2.1 Load and Inspect the Prepared Data

Before running the notebook, download and prepare the RealWaste dataset using the following command from the repository root:

```bash
python project_baselines/project_03/scripts/prepare_realwaste.py --download --seed 42
```

The preparation script creates a reproducible dataset with **training, validation, and test partitions**.

It also generates a **manifest** (`manifest.csv`) that describes the dataset. Each row corresponds to one image and contains:

- `image_path`: The location of the image file.
- `label`: The waste category assigned to the image.
- `split`: The dataset partition (`train`, `validation`, or `test`).

We will first load this manifest into a Pandas DataFrame. This allows us to explore the dataset and its class distribution without loading all the images into memory.

**Questions to consider:**
- How many images are available for each category?
- Are the classes balanced?
- How are the images distributed across the training, validation, and test partitions?


In [ ]:

import hashlib
import os
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

SEED = 42

# 1. Locate the project directory
# The notebook can be run from the repository root or the project folder.
project_dir = Path.cwd()
if not (project_dir / "data" / "prepared" / "realwaste").exists():
    project_dir = Path("project_baselines/project_03")

# Locate the prepared dataset.
# REALWASTE_PREPARED_DIR can optionally specify a different location.
data_dir = Path(
    os.environ.get(
        "REALWASTE_PREPARED_DIR",
        project_dir / "data" / "prepared" / "realwaste"
    )
).expanduser().resolve()


# 2. Load the dataset manifest
# Each row describes an image, including its category and dataset split.
manifest_path = data_dir / "manifest.csv"
images = pd.read_csv(manifest_path)

# The manifest stores image paths relative to the prepared data directory.
# Convert them to full paths so we can open the images later.
images["path"] = images["image_path"].map(
    lambda value: (data_dir / value).resolve()
)

# 3. Verify and inspect the dataset
# Check that every image referenced by the manifest actually exists.
assert images["path"].map(Path.is_file).all(), "Some source images are missing."

print(f"Images: {len(images)}")
print(f"Classes: {images['label'].nunique()}")

# Examine the first few entries.
display(images.head())

In [ ]:
# Inspect images per category and split
pd.crosstab(images["label"], images["split"], margins=True)

#### Inspecting Sample Images

Let's visualize one randomly selected image from each of the nine waste categories.

In [ ]:

# Get the nine waste categories in alphabetical order
classes = sorted(images["label"].unique())

# Create a 3×3 grid to display one example per category
fig, axes = plt.subplots(3, 3, figsize=(10, 10))

for axis, label in zip(axes.flat, classes):

    # Randomly select one image from the current category
    # Using SEED ensures we get the same examples on every run.
    example = images[images["label"] == label].sample(
        1, random_state=SEED
    ).iloc[0]

    # Open and display the image
    with Image.open(example["path"]) as image:
        axis.imshow(image.convert("RGB"))

    # Show the category name and hide the plot axes
    axis.set_title(label)
    axis.axis("off")

plt.tight_layout()
plt.show()

### 2.2 Frozen MobileNetV3-Small Features

Instead of training an image classifier from scratch, we use a **pretrained MobileNetV3-Small** model to extract numerical representations of our waste images.

The model was previously trained on ImageNet. We use its learned representations without updating its parameters (*frozen feature extraction*). Each image is converted into a **576-dimensional feature vector**, which we then use to train a Logistic Regression classifier.

#### Download Pretrained Model Weights

The model requires pretrained weights (approximately 10 MB). Torchvision automatically downloads these weights the first time the model is initialized, unless they are already available in the local PyTorch cache.

**If your notebook cannot access the internet**, download the weights beforehand from an internet-connected terminal:

```bash
python -c "
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights
mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.DEFAULT)
"
```

#### Preparing Images for Feature Extraction

Before extracting image embeddings, we need to prepare the images in the format expected by MobileNetV3-Small.

We use three PyTorch components:

- **Preprocessing:** Applies the resizing, cropping, and normalization required by the pretrained model.
- **Dataset:** Defines how to load and preprocess one image at a time.
- **DataLoader:** Groups images into batches so the model can process them efficiently.

We keep the images in their original dataset order (`shuffle=False`). This ensures that the extracted embeddings remain aligned with the corresponding image paths and labels.

At this stage, we are only preparing the input pipeline. The pretrained model will be loaded and used to extract features in the next step.


In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision.models import MobileNet_V3_Small_Weights, mobilenet_v3_small

# 1. Configure the pretrained image encoder
torch.manual_seed(SEED)

# Use a GPU when available; otherwise, run on the CPU.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Select pretrained ImageNet weights and their corresponding preprocessing.
# This includes resizing, cropping, normalization, and conversion to tensors.
# The weights will be downloaded later, when we initialize MobileNet.
weights = MobileNet_V3_Small_Weights.DEFAULT
preprocess = weights.transforms()


# 2. Define how to load and preprocess individual images
class ManifestImages(Dataset):
    def __init__(self, frame):
        # Keep the images in the same order as our dataset manifest.
        self.frame = frame.reset_index(drop=True)

    def __len__(self):
        # Number of images in the dataset.
        return len(self.frame)

    def __getitem__(self, index):
        # Read an image using the file path in the manifest.
        with Image.open(self.frame.loc[index, "path"]) as image:
            # Convert to RGB and apply MobileNet's expected preprocessing.
            tensor = preprocess(image.convert("RGB"))

        return tensor


# 3. Create batches for efficient feature extraction
# shuffle=False preserves the order of the manifest, allowing us to
# associate each extracted feature vector with its original image.
loader = DataLoader(
    ManifestImages(images),
    batch_size=32,
    shuffle=False,
    num_workers=0
)

print("Feature-extraction device:", device)
print("Images to process:", len(images))
print("Number of batches:", len(loader))


### 2.3 Extracting Image Embeddings with MobileNetV3-Small

In our text-classification exercises, we used techniques such as TF-IDF to transform text into numerical feature vectors. We will now do something similar for images.

Instead of designing image features manually, we use **MobileNetV3-Small**, a convolutional neural network pretrained on ImageNet, to extract meaningful visual representations.

**How does it work?**

1. **Pretrained model:** MobileNetV3-Small has already learned visual patterns from a large collection of images.
2. **Remove the classification head:** We use the network's feature-extraction layers, rather than its original ImageNet classifier.
3. **Extract embeddings:** Each image is transformed into a vector of **576 numerical features**, capturing information about its visual appearance.
4. **Train our own classifier:** In the next step, we will use these features as inputs to a Logistic Regression model trained to recognize the nine RealWaste categories.

We keep MobileNet's parameters **frozen**: its learned weights are not updated using our waste images. Only the Logistic Regression classifier will be trained on our dataset.

#### Why cache the embeddings?

Processing thousands of images through a neural network can take time. Since the feature extractor is frozen, it produces the same embeddings for the same images.

We therefore save the extracted features in `mobilenet_v3_small_features.npz` and reuse them on subsequent runs.

The notebook checks whether the cached features match the dataset manifest and model configuration. If they do, we can skip the feature-extraction step.

**Important:** Image embeddings are not class predictions. They are numerical representations that our classifier will learn to associate with waste categories.


In [ ]:
import hashlib
import time
import numpy as np

# 1. Check whether we already extracted the image features
cache_path = data_dir / "mobilenet_v3_small_features.npz"

# Record which dataset manifest and pretrained model were used.
# This helps us avoid reusing features from a different dataset or model.
manifest_sha256 = hashlib.sha256(manifest_path.read_bytes()).hexdigest()
encoder_id = f"mobilenet_v3_small:{weights}"

features = None

if cache_path.exists():
    cached = np.load(cache_path, allow_pickle=False)

    # Reuse the cache only if it matches our current configuration.
    cache_matches = (
        {"features", "manifest_sha256", "encoder_id"}.issubset(cached.files)
        and cached["manifest_sha256"].item() == manifest_sha256
        and cached["encoder_id"].item() == encoder_id
    )

    if cache_matches:
        features = cached["features"]
        print("Loaded cached features:", cache_path)


# 2. Extract the features if no valid cache is available
if features is None:

    # Load MobileNetV3-Small with pretrained ImageNet weights.
    # Torchvision downloads the weights automatically if they are not cached.
    network = mobilenet_v3_small(weights=weights)

    # Keep only the feature-extraction part of the network:
    # - features: convolutional layers that recognize visual patterns
    # - avgpool: summarizes the spatial feature maps
    # - Flatten: converts the result into a 576-dimensional vector
    #
    # We deliberately exclude the original ImageNet classification head.
    encoder = nn.Sequential(
        network.features,
        network.avgpool,
        nn.Flatten(1)
    ).to(device).eval()

    # Freeze the model: we will not update its parameters.
    for parameter in encoder.parameters():
        parameter.requires_grad_(False)

    # Process the images batch by batch.
    batches = []
    started = time.perf_counter()

    # inference_mode disables gradient computation, saving memory and time.
    with torch.inference_mode():
        for batch in loader:
            # Extract features on the selected device (GPU or CPU).
            embeddings = encoder(batch.to(device))

            # Move results to CPU and convert them to NumPy arrays.
            batches.append(embeddings.cpu().numpy())

    # Combine all batches into one matrix:
    # one row per image, 576 features per row.
    features = np.vstack(batches).astype("float32")

    # Save features for reuse in later experiments.
    np.savez_compressed(
        cache_path,
        features=features,
        manifest_sha256=np.array(manifest_sha256),
        encoder_id=np.array(encoder_id)
    )

    print(f"Extracted and cached features in {time.perf_counter() - started:.1f} seconds")


# 3. Verify the resulting feature matrix
assert features.shape == (len(images), 576)
print("Embedding matrix:", features.shape)


### 2.4 Logistic Regression Baseline

We now train a **Logistic Regression classifier** to recognize the nine RealWaste categories using the image embeddings extracted by MobileNetV3-Small.

Each image is represented by a 576-dimensional feature vector. Rather than training the entire neural network, we keep the image encoder frozen and train only Logistic Regression to associate these features with waste categories.

We use the dataset partitions as follows:

- **Training:** Fit the classifier using the image embeddings and their labels.
- **Validation:** Generate predictions and class probabilities to evaluate the model.
- **Test:** Keep this partition separate for final evaluation.

This approach is computationally inexpensive and allows us to experiment with different classifiers or training strategies without extracting the image features again.


In [ ]:
from sklearn.linear_model import LogisticRegression

# 1. Select the training and validation partitions
# These masks identify which rows of the embedding matrix belong to each split.
train_mask = images["split"].eq("train").to_numpy()
validation_mask = images["split"].eq("validation").to_numpy()

# 2. Train Logistic Regression on the image embeddings
# Each training example has 576 input features and one waste-category label.
classifier = LogisticRegression(max_iter=2000, random_state=SEED)

classifier.fit(
    features[train_mask],
    images.loc[train_mask, "label"]
)

# 3. Generate predictions on the validation set
validation_predictions = classifier.predict(features[validation_mask])

# Obtain the estimated probability for each of the nine categories.
# These will also be useful for investigating model uncertainty later.
validation_probabilities = classifier.predict_proba(features[validation_mask])


### 2.5 Initial Evaluation

Let's evaluate our classifier on the **validation set**, which was not used for training.

We start with two complementary metrics:

- **Accuracy:** The proportion of images classified correctly across all categories.
- **Macro-F1:** The average F1-score across the nine categories, giving each category equal importance regardless of its frequency.

Since RealWaste contains different numbers of images per category, accuracy alone may hide poor performance on less frequent classes.

We also examine a **classification report** showing precision, recall, and F1-score for each category.

**Questions to consider:**
- Does the model perform equally well across all categories?
- Which categories are particularly difficult to recognize?
- What might explain the differences in performance?

These results provide a starting point for investigating the model's strengths and limitations. We keep the test set separate for final evaluation.


In [ ]:
from sklearn.metrics import accuracy_score, classification_report, f1_score

# Get the true labels for the validation images
validation_labels = images.loc[validation_mask, "label"]

# 1. Overall performance
# Accuracy: fraction of correctly classified images
accuracy = accuracy_score(validation_labels, validation_predictions)

# Macro-F1: compute F1 for each class, then average equally
macro_f1 = f1_score(
    validation_labels,
    validation_predictions,
    average="macro"
)

print(f"Accuracy: {accuracy:.3f}")
print(f"Macro-F1: {macro_f1:.3f}")


# 2. Per-class performance
# Examine precision, recall, F1-score, and support for each category.
print(classification_report(
    validation_labels,
    validation_predictions,
    labels=classifier.classes_,
    zero_division=0
))

## 3. Your experiments